In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys

BASE_DIR = "/content/drive/MyDrive/Informs_competition"
MYCODE_DIR = f"{BASE_DIR}/Mycode"
DATA_DIR = f"{BASE_DIR}/data"

# 加入路径
sys.path.append(MYCODE_DIR)

# 测试：列出 Mycode 文件夹内容
!ls $MYCODE_DIR


demo_evaluate.py  demo_sarima.py  demo_seq2seq.py  __pycache__


In [ ]:
import demo_seq2seq
import demo_evaluate

In [ ]:
import xarray as xr
ds = xr.open_dataset(f"{DATA_DIR}/train.nc")
print(ds)

<xarray.Dataset> Size: 159MB
Dimensions:    (location: 83, timestamp: 2161, feature: 109)
Coordinates:
  * location   (location) <U5 2kB '26001' '26003' '26005' ... '26163' '26165'
  * feature    (feature) object 872B 'SBT113' 'SBT114' 'SBT123' ... 'wz' 'wz_1'
  * timestamp  (timestamp) datetime64[ns] 17kB 2023-04-01 ... 2023-06-30
    state      (location) <U2 664B ...
Data variables:
    tracked    (location, timestamp) float64 1MB ...
    out        (location, timestamp) float64 1MB ...
    weather    (location, timestamp, feature) float64 156MB ...
Attributes:
    time_start:  2022-01-01T00:00:00
    time_end:    2022-01-31T23:00:00
    time_now:    2025-07-08T14:59:10


**Data Cleaning**

In [ ]:
import numpy as np

weather = ds['weather'].values  # (location, timestamp, feature)
print("Shape:", weather.shape)

# 缺失值比例
missing_ratio = np.isnan(weather).mean()
print("缺失值比例:", missing_ratio)

Shape: (83, 2161, 109)
缺失值比例: 0.0


In [ ]:
import numpy as np
import pandas as pd

# 把 weather 重新拿出来（如果你之前已经有 ds 就不用这句）
# weather = ds['weather'].values   # shape (location, timestamp, feature)

# 把三维数据展平成二维 (样本数, 特征数)
n_loc, n_time, n_feat = weather.shape
weather_2d = weather.reshape(-1, n_feat)

# 转成 DataFrame，方便统计
feature_names = ds['feature'].values
df_weather = pd.DataFrame(weather_2d, columns=feature_names)

# 统计每个特征的负数个数和比例
neg_stats = (df_weather < 0).sum().to_frame("neg_count")
neg_stats["total_count"] = len(df_weather)
neg_stats["neg_ratio"] = neg_stats["neg_count"] / neg_stats["total_count"]

# 按负值比例排序
neg_stats = neg_stats.sort_values("neg_ratio", ascending=False)

print(neg_stats.head(20))  # 先看前20个负数最多的

           neg_count  total_count  neg_ratio
cpofp         179197       179363   0.999075
unknown_3     170113       179363   0.948429
refd          166448       179363   0.927995
refc          154853       179363   0.863350
vstm          128691       179363   0.717489
v10            95165       179363   0.530572
v              92509       179363   0.515764
wz             92250       179363   0.514320
u              91707       179363   0.511293
wz_1           91417       179363   0.509676
vvcsh          90404       179363   0.504028
u10            89802       179363   0.500672
ishf           81229       179363   0.452875
ustm           79495       179363   0.443207
vucsh          70285       179363   0.391859
gflux          60506       179363   0.337338
cin            19011       179363   0.105992
lftx4          13966       179363   0.077864
lftx           10636       179363   0.059299
slhtf           5003       179363   0.027893


In [ ]:
import pandas as pd
import numpy as np

# 需要检查的变量
must_non_negative = [
    "prate", "tp", "crain", "csnow",   # 降水
    "r2", "sh2",                       # 湿度
    "tcc", "hcc", "mcc", "lcc",        # 云量
    "sdswrf", "sdlwrf", "suswrf", "sulwrf"  # 辐射
]

# 统计负值数量
neg_summary = []
for var in must_non_negative:
    if var in ds['feature'].values:  # 确认变量在数据里
        values = ds['weather'].sel(feature=var).values.flatten()
        neg_count = np.sum(values < 0)
        total_count = values.size
        neg_ratio = neg_count / total_count
        neg_summary.append((var, neg_count, total_count, neg_ratio))

# 转成 DataFrame 展示
neg_df = pd.DataFrame(neg_summary, columns=["variable", "neg_count", "total_count", "neg_ratio"])
print(neg_df.sort_values("neg_ratio", ascending=False))

   variable  neg_count  total_count  neg_ratio
0     prate          0       179363        0.0
1        tp          0       179363        0.0
2     crain          0       179363        0.0
3     csnow          0       179363        0.0
4        r2          0       179363        0.0
5       sh2          0       179363        0.0
6       tcc          0       179363        0.0
7       hcc          0       179363        0.0
8       mcc          0       179363        0.0
9       lcc          0       179363        0.0
10   sdswrf          0       179363        0.0
11   sdlwrf          0       179363        0.0
12   suswrf          0       179363        0.0
13   sulwrf          0       179363        0.0


In [ ]:
import numpy as np
import pandas as pd

# 取 weather 的三维数组
weather = ds['weather'].values  # (location, timestamp, feature)
num_locs, num_times, num_feats = weather.shape

# 展平到 2D (样本, 特征)，方便统计
weather_2d = weather.reshape(-1, num_feats)

# 统计每个特征的基本信息
stats = []
for i, feat in enumerate(ds['feature'].values):
    vals = weather_2d[:, i]
    mean = np.mean(vals)
    std = np.std(vals)
    min_val = np.min(vals)
    max_val = np.max(vals)
    p1 = np.percentile(vals, 1)
    p99 = np.percentile(vals, 99)
    stats.append([feat, mean, std, min_val, max_val, p1, p99])

df_stats = pd.DataFrame(stats, columns=["feature", "mean", "std", "min", "max", "p1", "p99"])
print(df_stats.head(20))  # 看前20个特征

   feature        mean         std    min          max          p1  \
0   SBT113  230.961661    8.293537    0.0   250.179993  218.756197   
1   SBT114  277.051781   17.734577    0.0   305.600006  222.899994   
2   SBT123  239.413582    9.168950    0.0   259.820007  221.449997   
3   SBT124  278.654294   17.791673    0.0   307.899994  225.356203   
4      aod    0.000000    0.000000    0.0     0.000000    0.000000   
5    bgrun    0.000000    0.000000    0.0     0.000000    0.000000   
6      blh  599.394865  591.961028    0.0  3401.091309   22.241575   
7     cape    4.375927   14.471057    0.0   212.600006    0.000000   
8   cape_1   39.035197  174.626664    0.0  4350.000000    0.000000   
9    cfnsf    0.000786    0.058836    0.0     5.000000    0.000000   
10   cfrzr    0.000000    0.000000    0.0     0.000000    0.000000   
11   cicep    0.000000    0.000000    0.0     0.000000    0.000000   
12     cin   -5.098477   26.340809 -627.0     4.612808 -140.000000   
13   cnwat    0.0239

In [ ]:
import numpy as np

def create_dataset_24h(ds, input_len=24, pred_len=24):
    """
    构造训练数据：输入过去 input_len 小时的 weather，预测未来 pred_len 小时的 out 总和
    """
    weather = ds['weather'].values   # (location, timestamp, feature)
    out = ds['out'].values           # (location, timestamp)
    n_loc, n_time, n_feat = weather.shape

    X_list, y_list = [], []
    for t in range(n_time - input_len - pred_len):
        # 输入窗口: 过去 input_len 小时的 weather
        X_list.append(weather[:, t:t+input_len, :])  # (location, input_len, feature)

        # 输出: 未来 pred_len 小时的 out 累积
        y_list.append(out[:, t+input_len : t+input_len+pred_len].sum(axis=1))  # (location,)

    X = np.array(X_list)  # (样本数, location, input_len, feature)
    y = np.array(y_list)  # (样本数, location)

    return X, y

# 生成训练数据
X_24, y_24 = create_dataset_24h(ds, input_len=24, pred_len=24)
print("X shape:", X_24.shape)
print("y shape:", y_24.shape)

X shape: (2113, 83, 24, 109)
y shape: (2113, 83)


In [ ]:
# 把 location 合并进 batch 维度
X_3d = X_24.reshape(-1, 24, 109)   # (2113*83, 24, 109)
y_3d = y_24.reshape(-1)            # (2113*83,)
print("X_3d:", X_3d.shape)
print("y_3d:", y_3d.shape)

X_3d: (175379, 24, 109)
y_3d: (175379,)


In [ ]:
!python {MYCODE_DIR}/demo_seq2seq.py \
    --data_path /content/drive/MyDrive/data/train.nc \
    --pred_len 24 \
    --save_path {BASE_DIR}/pred_24.csv \
    --epochs 2 \
    --batch_size 16

!python {MYCODE_DIR}/demo_seq2seq.py \
    --data_path /content/drive/MyDrive/data/train.nc \
    --pred_len 48 \
    --save_path {BASE_DIR}/pred_48.csv \
    --epochs 2 \
    --batch_size 16

Traceback (most recent call last):
  File "/content/drive/MyDrive/Informs_competition/Mycode/demo_seq2seq.py", line 238, in <module>
    main()
  File "/content/drive/MyDrive/Informs_competition/Mycode/demo_seq2seq.py", line 189, in main
    ds_train = open_ds(os.path.join(SPLIT_DIR, "train.nc"))
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/Informs_competition/Mycode/demo_seq2seq.py", line 33, in open_ds
    raise FileNotFoundError(path)
FileNotFoundError: data/train.nc
Traceback (most recent call last):
  File "/content/drive/MyDrive/Informs_competition/Mycode/demo_seq2seq.py", line 238, in <module>
    main()
  File "/content/drive/MyDrive/Informs_competition/Mycode/demo_seq2seq.py", line 189, in main
    ds_train = open_ds(os.path.join(SPLIT_DIR, "train.nc"))
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/Informs_competition/Mycode/demo_seq2seq.py", line 33, in open_ds
    raise FileNotFoundErr